In [0]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

import os 
import sys

project_pth = os.path.join(os.getcwd(),'..','..')
sys.path.append(project_pth)

from utils.transformations import reusable

## **DimUser**

#### **AUTOLOADER**

In [0]:
df_user = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimUser/chekpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageazureproject.dfs.core.windows.net/DimUser")

In [0]:
display(df_user)

user_id,user_name,country,subscription_type,start_date,end_date,updated_at,_rescued_data
1,Carlos Berry,Switzerland,Premium,2023-10-17,null,2025-09-23T19:49:55.000Z,null
2,Amanda Jenkins,Montserrat,Family,2024-09-28,null,2025-09-29T19:49:55.000Z,null
3,Daniel Cook,Chile,Premium,2025-07-07,null,2025-09-08T19:49:55.000Z,null
4,Peter Hernandez,Nigeria,Free,2024-07-16,null,2025-09-29T19:49:55.000Z,null
5,Yolanda Morris,Aruba,Premium,2025-05-12,null,2025-09-17T19:49:55.000Z,null
6,Stephen Murphy,Cook Islands,Free,2025-06-17,null,2025-09-12T19:49:55.000Z,null
7,Anthony Andrews Jr.,Libyan Arab Jamahiriya,Free,2024-06-08,null,2025-09-20T19:49:55.000Z,null
8,Felicia Jones,Haiti,Family,2024-09-20,null,2025-09-08T19:49:55.000Z,null
9,Hector Decker,Palau,Family,2024-01-08,null,2025-09-29T19:49:55.000Z,null
10,Frank Davis,New Zealand,Family,2023-12-13,null,2025-09-22T19:49:55.000Z,null


In [0]:
df_user = df_user.withColumn("user_name",upper(col("user_name")))
display(df_user)

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user,['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])
display(df_user)

user_id,user_name,country,subscription_type,start_date,end_date,updated_at
148,CHASE MOORE,Zambia,Family,2024-03-13,null,2025-09-18T19:49:55.000Z
463,JESSICA LEWIS,Slovenia,Free,2025-03-08,null,2025-10-06T19:49:55.000Z
471,JOSHUA BUTLER,Ethiopia,Free,2025-09-14,null,2025-09-25T19:49:55.000Z
496,ANDREW MATTHEWS,Taiwan,Family,2025-08-08,null,2025-09-17T19:49:55.000Z
243,MARTHA MARTIN,Chad,Premium,2024-05-11,null,2025-10-06T19:49:55.000Z
392,JENNIFER VILLANUEVA,Serbia,Free,2025-02-14,null,2025-10-06T19:49:55.000Z
31,TAMMY COOK,Bolivia,Premium,2025-02-15,null,2025-10-02T19:49:55.000Z
85,LISA SHERMAN,Japan,Family,2024-01-31,null,2025-09-11T19:49:55.000Z
137,AMY WEEKS,Sao Tome and Principe,Family,2024-07-02,null,2025-09-09T19:49:55.000Z
251,JERRY BOOTH,Lithuania,Family,2023-11-04,null,2025-09-17T19:49:55.000Z


In [0]:
df_user.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimUser/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@storageazureproject.dfs.core.windows.net/DimUser/data")\
            .toTable("spotify_cata.silver.DimUser")

## **DimArtist**

In [0]:
df_art = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimArt/chekpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageazureproject.dfs.core.windows.net/DimArtist")

In [0]:
display(df_art)

In [0]:
df_art_obj = reusable()

df_art = df_art_obj.dropColumns(df_art,['_rescued_data'])
df_art = df_art.dropDuplicates(['artist_id'])
display(df_art)   

artist_id,artist_name,genre,country,updated_at
148,Dawn Graham,Hip-Hop,Anguilla,2025-10-05T19:49:55.000Z
463,Michael Irwin,Pop,Tokelau,2025-09-23T19:49:55.000Z
471,Alex Mcclure,Jazz,Reunion,2025-09-13T19:49:55.000Z
496,Tim Tucker,Electronic,China,2025-10-05T19:49:55.000Z
243,Leslie Holt,Jazz,Myanmar,2025-09-20T19:49:55.000Z
392,Jason Harrell,Jazz,French Polynesia,2025-10-07T19:49:55.000Z
31,Alexandra Decker,Hip-Hop,Liechtenstein,2025-09-10T19:49:55.000Z
85,Douglas White,Hip-Hop,Antigua and Barbuda,2025-09-15T19:49:55.000Z
137,Charles Perry,Electronic,Zimbabwe,2025-10-04T19:49:55.000Z
251,Mary Moore,Hip-Hop,Bahamas,2025-10-03T19:49:55.000Z


In [0]:
df_art.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimArt/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@storageazureproject.dfs.core.windows.net/DimArt/data")\
            .toTable("spotify_cata.silver.DimArtist")

## **DimTrack**

In [0]:
df_track = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimTrack/chekpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageazureproject.dfs.core.windows.net/DimTrack")

In [0]:
display(df_track)

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,_rescued_data
1,Stand-alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,null
2,Programmable contextually-based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,null
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,null
4,Mandatory user-facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,null
5,Multi-layered needs-based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,null
6,Synchronized high-level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,null
7,Total human-resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,null
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,null
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,null
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,null


In [0]:
df_track = df_track.withColumn("durationFlag",when(col('duration_sec')<150,"low")\
                                            .when(col('duration_sec')<300,"medium")\
                                            .otherwise("high"))

df_track = df_track.withColumn("track_name",regexp_replace(col('track_name'),'-',' '))

df_track = reusable().dropColumns(df_track,['_rescued_data'])

display(df_track)

track_id,track_name,artist_id,album_name,duration_sec,release_date,updated_at,durationFlag
1,Stand alone intangible encryption,434,Technology Album,234,2021-02-09,2025-09-12T19:49:55.000Z,medium
2,Programmable contextually based forecast,418,Current Album,110,2023-09-30,2025-09-26T19:49:55.000Z,low
3,Enhanced tertiary Internet solution,35,Born Album,195,2022-01-11,2025-09-11T19:49:55.000Z,medium
4,Mandatory user facing framework,54,Ball Album,167,2023-06-30,2025-10-02T19:49:55.000Z,medium
5,Multi layered needs based concept,340,Doctor Album,137,2022-06-08,2025-10-02T19:49:55.000Z,low
6,Synchronized high level complexity,384,City Album,250,2021-10-16,2025-09-22T19:49:55.000Z,medium
7,Total human resource structure,317,Parent Album,194,2022-08-02,2025-10-07T19:49:55.000Z,medium
8,Exclusive multimedia matrices,37,Owner Album,206,2022-06-22,2025-10-02T19:49:55.000Z,medium
9,Digitized multimedia hardware,165,Hand Album,342,2023-12-03,2025-10-02T19:49:55.000Z,high
10,Phased dynamic utilization,15,Hair Album,199,2023-12-17,2025-10-07T19:49:55.000Z,medium


In [0]:
df_track.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimTrack/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@storageazureproject.dfs.core.windows.net/DimTrack/data")\
            .toTable("spotify_cata.silver.DimTrack")

%md
## **DimDate**

In [0]:
df_date = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimDate/chekpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageazureproject.dfs.core.windows.net/DimDate")

In [0]:
df_date = reusable().dropColumns(df_date,['_rescued_data'])

df_date.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@storageazureproject.dfs.core.windows.net/DimDate/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@storageazureproject.dfs.core.windows.net/DimDate/data")\
            .toTable("spotify_cata.silver.DimDate")

%md
## **FactStream**

In [0]:
df_fact = spark.readStream.format("cloudFiles")\
            .option("cloudFiles.format","parquet")\
            .option("cloudFiles.schemaLocation","abfss://silver@storageazureproject.dfs.core.windows.net/FactStream/chekpoint")\
            .option("schemaEvolutionMode","addNewColumns")\
            .load("abfss://bronze@storageazureproject.dfs.core.windows.net/FactStream")

In [0]:
display(df_fact)

stream_id,user_id,track_id,date_key,listen_duration,device_type,stream_timestamp,_rescued_data
1,361,74,20250518,156,Smart Speaker,2025-09-30T19:49:55.000Z,null
2,321,288,20250519,47,Mobile,2025-09-27T19:49:55.000Z,null
3,275,340,20250307,214,Mobile,2025-10-03T19:49:55.000Z,null
4,43,373,20250216,14,Desktop,2025-10-04T19:49:55.000Z,null
5,319,95,20250421,266,Desktop,2025-09-27T19:49:55.000Z,null
6,52,31,20250130,317,Desktop,2025-10-02T19:49:55.000Z,null
7,5,354,20250824,90,Mobile,2025-10-01T19:49:55.000Z,null
8,115,386,20250413,159,Smart Speaker,2025-09-29T19:49:55.000Z,null
9,439,95,20250930,290,Mobile,2025-10-04T19:49:55.000Z,null
10,40,389,20250227,53,Mobile,2025-10-01T19:49:55.000Z,null


In [0]:
df_fact = reusable().dropColumns(df_fact,['_rescued_data'])

df_fact.writeStream.format("delta")\
            .outputMode("append")\
            .option("checkpointLocation","abfss://silver@storageazureproject.dfs.core.windows.net/FactStream/chekpoint")\
            .trigger(once=True)\
            .option("path","abfss://silver@storageazureproject.dfs.core.windows.net/FactStream/data")\
            .toTable("spotify_cata.silver.FactStream")